<a href="https://colab.research.google.com/github/busycaesar/LLM_Eval/blob/Master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets anthropic tqdm pandas

## 1. Dataset Loading and Prompt Building `Configure`

In [ ]:
DATASET = "cais/mmlu"
SAMPLE_SIZE = 10

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET, "all", split="test")

if SAMPLE_SIZE:
    dataset = dataset.shuffle(seed=0).select(range(min(SAMPLE_SIZE, len(dataset))))

rows = [dict(r) for r in dataset]
print(f"{len(rows)} items loaded")
print(dataset.features)

In [ ]:
import re

LETTERS = ["A", "B", "C", "D"]

def build_prompt(row):
    choices = "\n".join(f"{LETTERS[i]}. {c}" for i, c in enumerate(row["choices"]))
    return (
        "Answer the following multiple choice question.\n\n"
        f"Question: {row['question']}\n\n"
        f"{choices}\n\n"
        "Reply with only the letter of the correct answer."
    )

def extract_prediction(response):
    match = re.search(r"\b([ABCD])\b", response.strip().upper())
    return match.group(1) if match else None

print(build_prompt(rows[0]))
print(extract_prediction("The correct answer is C."))

## 2. Model Loading `Configure`

In [ ]:
MODEL = "claude-sonnet-5"

In [ ]:
from google.colab import userdata
from anthropic import Anthropic

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
client = Anthropic(api_key=ANTHROPIC_API_KEY)

def infer_llm(prompt):
  response = client.messages.create(
    model=MODEL,
    max_tokens=16,
    messages=[{
      "role": "user",
      "content": prompt
    }],
  )

  return "".join(b.text for b in response.content if b.type == "text")

## 3. Helper functions

In [ ]:
import time

def call_model(prompt, retries=4):
    for attempt in range(retries):
        try:
            return infer_llm(prompt)
        except Exception as e:
            if attempt == retries - 1:
                return f"ERROR: {e}"
            time.sleep(2 ** attempt)

In [ ]:
def run_evaluation(row):
    prompt = build_prompt(row)
    raw_response = call_model(prompt)
    prediction = extract_prediction(raw_response)
    correct_answer = LETTERS[row["answer"]]

    return {
        "subject": row["subject"],
        "question": row["question"],
        "raw_response": raw_response,
        "prediction": prediction,
        "correct_answer": correct_answer,
        "accurate": prediction == correct_answer,
    }

## 4. Run the evals

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

with ThreadPoolExecutor(max_workers=8) as ex:
    results = list(tqdm(ex.map(run_evaluation, rows), total=len(rows)))

## 5. Analyze the evals result

In [ ]:
import pandas

results_table = pandas.DataFrame(results)
results_table.head()

accuracy = results_table["accurate"].mean()
# Counts rows where the prediction is not present maybe due to failed LLM call or no valid answer present in the response.
unparsed_responses = results_table["prediction"].isna().sum()
# Counts rows where the LLM call failed.
errors = results_table["raw_response"].str.startswith("ERROR:").sum()


print(f"Model:      {MODEL}")
print(f"Dataset:    {DATASET}")
print(f"Items:      {len(results_table)}")
print(f"Accuracy:   {accuracy:.3f}")
# If errors == unparsed_responses, it indicates that because the LLM call failed, the correct answer could not be predicted.
print(f"Unparsed:   {unparsed_responses}")
print(f"API errors: {errors}")

## 6. Save eval results

In [ ]:
results_table.to_csv(f"results_{MODEL.replace("/", "_")}.csv", index=False)